# 13 — Landmark finder per 5km grid (test cases)

**Problem.** The info-sheet Google Maps link points at the 500m *centroid*, which sits in an empty field with no routable road — so road navigation fails. But enumerators in TZ navigate by **named places**: villages, primary schools, churches, dispensaries, markets.

**This notebook.** For a few test **5km grid cells**, it finds the recognisable OSM landmarks near each selected 500m sub-cell and ranks them **nearest-first**, with the straight-line distance from the cell edge and a **compass direction** from the cell. Output: a ranked table (with clickable Google Maps links) + a little map per sub-cell.

The heavy lifting lives in [`src/data_processing/landmarks.py`](../src/data_processing/landmarks.py) (Overpass query across mirrors, ranking, caching). Results are cached to a CSV, so re-running is instant and offline.

> Once you're happy with a few cells here, the same function runs over **all** sub-cells via `python scripts/build_landmarks.py`, and the cache feeds the HTML info sheets.

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import geopandas as gpd
import pandas as pd
from IPython.display import HTML

from src.utils.config_loader import load_config, get_data_dir
from src.data_processing.load_boundaries import load_control_grid, load_selected_subcells
from src.data_processing.landmarks import build_subcell_landmarks, category_label

config = load_config()
data_dir = get_data_dir(config)

grid_5km = load_control_grid(data_dir)
selected = load_selected_subcells(data_dir)
print(f"\n{len(grid_5km)} control 5km cells | {len(selected)} selected 500m sub-cells")
print("5km cells available:", sorted(selected['5km_id'].dropna().astype(int).unique().tolist()))

## Pick your test 5km cells

Edit `TEST_5KM_IDS`. The defaults are chosen to span the coverage range:

| 5km id | what to expect |
|--------|----------------|
| `10758` (Ibumu) | well-covered — a school **in** the cell + Isaka village next door |
| `11299` (Wotta) | medium — a primary school ~100 m out |
| `12213` (Godegode) | sparse — nearest named village ~3.5 km away (shows the fallback) |

In [ ]:
TEST_5KM_IDS = [10758, 11299, 12213]   # <-- edit me

test_sub = selected[selected['5km_id'].fillna(-1).astype(int).isin(TEST_5KM_IDS)].copy()
print(f"{len(test_sub)} sub-cells across {test_sub['5km_id'].nunique()} test 5km cells")
test_sub[['grid_id', '5km_id', 'ward_name', 'selection_role', 'building_count',
          'latitude', 'longitude']]

## Find landmarks (live Overpass, cached)

This queries OpenStreetMap **once per 5km cell** (the sub-cells inside a cell overlap, so one query at a 10 km radius covers them all) and reuses the result for every sub-cell.

⚠️ **The public Overpass servers are slow** — expect roughly **60–90 s per 5km cell** on a cold run (they frequently make you wait out the timeout). So the first run over these 3 test cells takes a few minutes. But results are **cached** to the CSV below, so:

- **This cache is already pre-warmed** for the 3 default cells → with `force=False` this cell returns **instantly** (reads the CSV, no network).
- Add new cells to `TEST_5KM_IDS` and only those are queried; the rest come from cache.
- Resumable: if a mirror fails mid-run, just run the cell again to fill in the gaps.
- Set `force=True` to re-query cells already cached.

In [ ]:
CACHE_PATH = PROJECT_ROOT / 'notebooks' / '_landmark_cache_test.csv'

lm = build_subcell_landmarks(
    test_sub,
    CACHE_PATH,
    force=False,        # set True to re-query
    max_landmarks=8,
    radius_m=10000,     # search radius around each 5km cell centre
    pause=0.5,
)

# keep just the test cells (cache may hold others from earlier runs)
lm = lm[lm['grid_id'].isin(test_sub['grid_id'])].copy()
print(f"\n{len(lm)} landmark rows for {lm['grid_id'].nunique()} sub-cells")

## Ranked landmark tables (clickable)

One block per 5km cell → each selected sub-cell → its landmarks nearest-first. `dist_m` is the straight-line distance from the **cell edge** (0 = inside the cell); `direction` is where the landmark lies **from** the cell.

In [ ]:
def dist_label(m):
    return 'in 500m cell' if m == 0 else (f"{m} m" if m < 1000 else f"{m/1000:.2f} km")

def link(url, text):
    return f'<a href="{url}" target="_blank">{text}</a>' if isinstance(url, str) and url else '—'

blocks = []
for km5 in TEST_5KM_IDS:
    cell_mask = test_sub['5km_id'].fillna(-1).astype(int) == km5
    sub_ids = test_sub.loc[cell_mask, 'grid_id']
    ward = test_sub.loc[cell_mask, 'ward_name'].dropna()
    ward = ward.iloc[0] if len(ward) else '?'
    blocks.append(f"<h3>5km grid cell <code>{km5}</code> — {ward}</h3>")
    for gid in sub_ids:
        rows = lm[lm['grid_id'] == gid].sort_values('rank')
        role = test_sub.loc[test_sub['grid_id'] == gid, 'selection_role'].iloc[0]
        blocks.append(f"<b>500m sub-cell {gid}</b> <i>({role})</i> "
                      f"&nbsp;·&nbsp; 5km grid <code>{km5}</code>")
        if len(rows) == 0:
            blocks.append("<div style='color:#b45309'>— no OSM landmarks found (even at 10 km) —</div>")
            continue
        disp = pd.DataFrame({
            '#': rows['rank'],
            '5km grid': km5,
            'distance': rows['dist_m'].map(dist_label),
            'dir': rows['direction'],
            'type': rows['category'].map(category_label),
            'name': rows['name'],
            'coordinates': [f"{la:.5f}, {lo:.5f}" for la, lo in zip(rows['latitude'], rows['longitude'])],
            'Google': rows['gmaps_url'].map(lambda u: link(u, 'pin ↗')),
            'OSM': rows['osm_url'].map(lambda u: link(u, 'attrs ↗')),
            'OSM map': rows['osm_map_url'].map(lambda u: link(u, 'map ↗')),
        })
        blocks.append(disp.to_html(index=False, escape=False))
HTML('\n'.join(blocks))

## Maps: sub-cell + ranked landmarks

Red square = the 500m sub-cell; red dot = its centroid (the old nav target). Landmarks are numbered by rank (1 = nearest) and coloured by type. Green outline means the landmark is inside the cell.

In [ ]:
import matplotlib.pyplot as plt
import contextily as cx
from shapely.geometry import Point

WM = 3857
GOOGLE_HYBRID = 'https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}'
CAT_COLOR = {'settlement': '#2563eb', 'school': '#c026d3', 'worship': '#7c3aed',
             'health': '#dc2626', 'market': '#ea580c', 'shop': '#ca8a04',
             'civic': '#0891b2', 'water': '#0d9488', 'other': '#6b7280'}

def plot_cell_landmarks(gid, ax=None):
    sq = test_sub[test_sub['grid_id'] == gid].to_crs(WM)
    if len(sq) == 0:
        return
    km5 = int(test_sub.loc[test_sub['grid_id'] == gid, '5km_id'].fillna(-1).iloc[0])
    rows = lm[lm['grid_id'] == gid].sort_values('rank')
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(8, 8))
    sq.boundary.plot(ax=ax, color='red', linewidth=2.5, zorder=5)
    cen = sq.geometry.centroid
    cen.plot(ax=ax, color='red', markersize=45, zorder=6)
    if len(rows):
        pts = gpd.GeoSeries([Point(xy) for xy in zip(rows['longitude'], rows['latitude'])],
                            crs=4326).to_crs(WM)
        for (rk, r), p in zip(rows.iterrows(), pts):
            col = CAT_COLOR.get(r['category'], '#6b7280')
            edge = '#16a34a' if r['inside'] else 'white'
            ax.scatter(p.x, p.y, s=170, c=col, edgecolors=edge, linewidths=2, zorder=7)
            ax.annotate(str(r['rank']), (p.x, p.y), color='white', fontsize=8,
                        ha='center', va='center', fontweight='bold', zorder=8)
            ax.annotate(f"  {r['name']} ({dist_label(r['dist_m'])})", (p.x, p.y),
                        color=col, fontsize=8, ha='left', va='center', zorder=8)
    # frame around the square + landmarks
    xs = list(sq.total_bounds[[0, 2]]);  ys = list(sq.total_bounds[[1, 3]])
    if len(rows):
        xs += list(pts.x); ys += list(pts.y)
    pad = max(max(xs) - min(xs), max(ys) - min(ys)) * 0.15 + 200
    ax.set_xlim(min(xs) - pad, max(xs) + pad)
    ax.set_ylim(min(ys) - pad, max(ys) + pad)
    try:
        cx.add_basemap(ax, source=GOOGLE_HYBRID)
    except Exception as e:
        print(f"basemap skipped for {gid}: {e}")
    role = test_sub.loc[test_sub['grid_id'] == gid, 'selection_role'].iloc[0]
    ax.set_title(f"5km grid {km5} · 500m sub-cell {gid} ({role}) — {len(rows)} landmarks",
                 fontsize=10)
    ax.set_axis_off()
    if own:
        plt.tight_layout(); plt.show()

for gid in test_sub['grid_id']:
    plot_cell_landmarks(gid)

## Run over everything & hand it to the field team

**1. Build the landmark cache for ALL selected sub-cells** (resumable — re-run if Overpass times out; ~60–90 s per 5km cell):

```bash
python scripts/build_landmarks.py
# writes 01_input_data/boundaries/subcell_landmarks.csv
```

**2. Deliverable A — one shareable dashboard** for Ravina (search + filter, per 500m & 5km cell, ranked landmarks, Google **and** OSM route/pin links, a Start-point box to route from wherever the enumerators are). Write it straight onto the shared drive so it's easily deployable:

```bash
python scripts/generate_landmark_dashboard.py \
    --output "G:\Shared drives\...\0_Listing\1_Input\listing_maps\landmark_dashboard.html"
```

It's a **single self-contained HTML file** — Ravina just double-clicks it (from the Drive or a copy). Landmarks and all route/pin links work anywhere; the overview maps + "full info sheet" links resolve when it sits next to the `listing_maps/` folders.

**3. Deliverable B — the per-cell info sheets** now include a "How to find each sub-cell — landmarks" section automatically (once the cache from step 1 exists):

```bash
python scripts/generate_info_sheets.py --output-dir "G:\...\1_Input\listing_maps"
```

Both read the same `subcell_landmarks.csv`, so they always agree.

## Happy with it? Run over everything

```bash
# build the landmark cache for ALL selected sub-cells (resumable — re-run if Overpass times out)
python scripts/build_landmarks.py

# or test the CLI on one sub-cell / a handful first
python scripts/build_landmarks.py --single G_0588_0987
python scripts/build_landmarks.py --limit 5
```

That writes `01_input_data/boundaries/subcell_landmarks.csv`, which the info sheets will read to render a **“How to find this cell”** section (ranked landmarks + directions) next to the existing maps.